# Data download and preparation (educational walk-through)


This notebook explains the data preparation steps that are later automated in:
```
scripts/download_data.py
scripts/prepare_data.py
```

The goal here is to understand why these steps are required before building TensorBoard Projector artifacts.

Originally dataset comes for Kaggle:
 - https://www.kaggle.com/datasets/miadul/animal-image-classification-5-species
 - https://www.kaggle.com/datasets/olafkrastovski/handwritten-digits-0-9


For this project the same datasets were uploaded to publicly availabe google drive storage.


## Step 1 - Download datasets

Download datasets for google drive (publicly available).

`gdown` package is used here.

In [10]:
# !pip install gdown

In [13]:
from pathlib import Path
import gdown


def download_file_public(file_id: str, output_path: str):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    url = f"https://drive.google.com/uc?id={file_id}"
    gdown.download(url, str(output_path), quiet=False)

In [14]:
download_file_public('14yuKcpUMqZjrlsymswOxEcdiIL2oLMyJ', '../animals.zip')

Downloading...
From: https://drive.google.com/uc?id=14yuKcpUMqZjrlsymswOxEcdiIL2oLMyJ
To: /Users/maksymstefanko/ML/ML-love/ml-image-vectors-vis-tensorboard/animals.zip
100%|██████████| 5.59M/5.59M [00:01<00:00, 3.78MB/s]


In [15]:
download_file_public('1XWbVjoOs60D7EW-zy2qVwYe2z3vm-jqN', '../digits.zip')

Downloading...
From (original): https://drive.google.com/uc?id=1XWbVjoOs60D7EW-zy2qVwYe2z3vm-jqN
From (redirected): https://drive.google.com/uc?id=1XWbVjoOs60D7EW-zy2qVwYe2z3vm-jqN&confirm=t&uuid=cac4ac40-29dc-4764-9ecd-421a2295e65d
To: /Users/maksymstefanko/ML/ML-love/ml-image-vectors-vis-tensorboard/digits.zip
100%|██████████| 69.9M/69.9M [00:10<00:00, 6.92MB/s]


## Step 2 - Unzip files

Extact downloaded archives data into folders.

This logic is also embedded into `download_data.py` so the repo can be set up with a single command

In [16]:
import zipfile

DATA_DIR = Path("../")

def unzip_file(zip_path: Path, extract_to: Path):
    print(f"Extracting {zip_path} → {extract_to}")
    extract_to.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_to)

In [17]:
unzip_file(DATA_DIR / 'animals.zip', Path('../images'))

Extracting ../animals.zip → ../images


In [18]:
unzip_file(DATA_DIR / 'digits.zip', Path('../digits'))

Extracting ../digits.zip → ../digits


## Step 3 - Why renaming is required

Real datasets contain:
 - nested folders (`train/test/validation`)
 - random filenames (`download (1).jpeg`, `images (7).jpeg`)
 - inconsistent ordering from os.walk

This breaks:
 - metadata ↔ embeddings alignment
 - sprite ↔ embeddings alignment
 - reproducibility across machines

## Step 4 - Demonstrate naive renaming

We detect folders that actually contain images - regardless of depth.

In [21]:
from pathlib import Path

EXTS = {".jpg", ".jpeg", ".png"}

root = Path("../images")

for folder in root.rglob("*"):
    if not folder.is_dir():
        continue

    images = [p for p in folder.iterdir() if p.suffix.lower() in EXTS]
    if not images:
        continue

    print(folder, "->", len(images))

../images/animals_dataset/test/cat -> 17
../images/animals_dataset/test/dog -> 16
../images/animals_dataset/test/deep -> 16
../images/animals_dataset/test/cow -> 16
../images/animals_dataset/test/lion -> 17
../images/animals_dataset/train/cat -> 91
../images/animals_dataset/train/dog -> 111
../images/animals_dataset/train/deep -> 87
../images/animals_dataset/train/cow -> 86
../images/animals_dataset/train/lion -> 89
../images/animals_dataset/validation/cat -> 17
../images/animals_dataset/validation/dog -> 17
../images/animals_dataset/validation/deep -> 17
../images/animals_dataset/validation/cow -> 15
../images/animals_dataset/validation/lion -> 17


## Step 5 - Renaming logic

This is the core logic moved into `scripts/prepare_data.py`

In [22]:
def rename_with_prefix(root: Path):
    for folder in root.rglob("*"):
        images = sorted([p for p in folder.iterdir() if p.suffix.lower() in EXTS]) if folder.is_dir() else []
        if not images:
            continue

        prefix = folder.name
        for i, p in enumerate(images, 1):
            tmp = folder / f"__tmp__{i}{p.suffix.lower()}"
            p.rename(tmp)

        for i, p in enumerate(sorted(folder.glob("__tmp__*")), 1):
            p.rename(folder / f"{prefix}_{i}{p.suffix.lower()}")

In [24]:
rename_with_prefix(Path("../images"))
rename_with_prefix(Path("../digits"))

## Step 6 - Limit digits dataset

Limit images count per class (folder) as projector might becomes unreadable with thousands of points


In [25]:
from pathlib import Path

root = Path("../digits")
MAX_PER_FOLDER = 100
EXTS = {".jpg", ".jpeg", ".png"}

for folder in sorted(root.iterdir()):
    if not folder.is_dir():
        continue

    images = sorted([p for p in folder.iterdir() if p.suffix.lower() in EXTS])

    if len(images) <= MAX_PER_FOLDER:
        print(f"{folder.name}: {len(images)} (ok)")
        continue

    to_delete = images[MAX_PER_FOLDER:]

    for p in to_delete:
        p.unlink()

    print(f"{folder.name}: removed {len(to_delete)}, kept {MAX_PER_FOLDER}")

print("Done.")

0: removed 2136, kept 100
1: removed 2141, kept 100
2: removed 2133, kept 100
3: removed 2102, kept 100
4: removed 2079, kept 100
5: removed 2026, kept 100
6: removed 2021, kept 100
7: removed 2016, kept 100
8: removed 1985, kept 100
9: removed 1916, kept 100
Done.


## Done

First two steps described in this notebook can be run with command:
```
python scripts/download_data.py
```

Reemain steps described in this notebook can be run with command:
```
python scripts/prepare_data.py
```

After that, it is safe to run.
```
python scripts/build_projector.py
```

Or find more details in `build_projector.ipynb` notebook